# 🧠 ADHD 코칭 AI 파인튜닝

ADHD 전문 코칭 AI 모델을 만들기 위한 파인튜닝 노트북입니다.

**사용 방법:**
1. 런타임 > 런타임 유형 변경 > GPU (T4) 선택
2. 셀을 순서대로 실행
3. 완료 후 모델 다운로드

## 1. 패키지 설치

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
print("설치 완료!")

## 2. 학습 데이터 업로드

좌측 파일 탭에서 `sft_dataset.json`과 `dpo_dataset.json`을 업로드하세요.

In [ ]:
from google.colab import files
import os

# 파일 업로드
print("sft_dataset.json 파일을 업로드하세요:")
uploaded = files.upload()

print("\ndpo_dataset.json 파일을 업로드하세요:")
uploaded = files.upload()

print("\n업로드 완료!")

## 3. 모델 로드

In [ ]:
from unsloth import FastLanguageModel
import torch

# 베이스 모델 선택 (한국어 성능 좋은 Qwen2 사용)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

print(f"모델 로드 완료! GPU 메모리: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# LoRA 어댑터 추가
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA 어댑터 추가 완료!")

## 4. 데이터셋 준비

In [ ]:
import json
from datasets import Dataset

# ADHD 코치 시스템 프롬프트
SYSTEM_PROMPT = """당신은 ADHD 전문 코치입니다. 피코치의 말에 공감하고,
ADHD 뇌의 특성을 이해하며, 작고 구체적인 실행 단계를 제안합니다.
완벽보다 진행을, 비판보다 격려를 우선합니다."""

# SFT 데이터 로드
with open('sft_dataset.json', 'r', encoding='utf-8') as f:
    sft_data = json.load(f)

# 채팅 형식으로 변환
def format_for_training(item):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["instruction"]},
        {"role": "assistant", "content": item["response"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

formatted_data = [format_for_training(item) for item in sft_data]
train_dataset = Dataset.from_list(formatted_data)

print(f"학습 데이터: {len(train_dataset)}개")
print(f"\n샘플 데이터:\n{formatted_data[0]['text'][:500]}...")

## 5. SFT 학습 (ADHD 지식 주입)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./adhd_coach_sft",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

print("SFT 학습 시작...")
trainer.train()
print("\nSFT 학습 완료!")

## 6. DPO 학습 (응답 스타일 정렬) - 선택사항

In [ ]:
from trl import DPOTrainer

# DPO 데이터 로드
with open('dpo_dataset.json', 'r', encoding='utf-8') as f:
    dpo_data = json.load(f)

# DPO 형식으로 변환
def format_for_dpo(item):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["prompt"]}
    ]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    return {
        "prompt": prompt,
        "chosen": item["chosen"],
        "rejected": item["rejected"]
    }

dpo_formatted = [format_for_dpo(item) for item in dpo_data]
dpo_dataset = Dataset.from_list(dpo_formatted)

print(f"DPO 데이터: {len(dpo_dataset)}개")

In [ ]:
dpo_args = TrainingArguments(
    output_dir="./adhd_coach_dpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    beta=0.1,
    max_length=2048,
    max_prompt_length=1024,
)

print("DPO 학습 시작...")
dpo_trainer.train()
print("\nDPO 학습 완료!")

## 7. 모델 테스트

In [ ]:
# 추론 모드로 전환
FastLanguageModel.for_inference(model)

def chat(user_message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 어시스턴트 응답만 추출
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

# 테스트
test_questions = [
    "할 일이 너무 많아서 뭐부터 해야 할지 모르겠어요",
    "또 미루고 말았어요. 저 왜 이럴까요?",
    "집중이 안 돼서 일을 시작하기가 너무 어려워요"
]

print("=" * 50)
print("ADHD 코칭 AI 테스트")
print("=" * 50)

for q in test_questions:
    print(f"\n🙋 피코치: {q}")
    print(f"\n🧑‍⚕️ 코치: {chat(q)}")
    print("-" * 50)

## 8. 모델 저장 및 다운로드

In [ ]:
# 모델 저장
model.save_pretrained("adhd_coach_final")
tokenizer.save_pretrained("adhd_coach_final")

print("모델 저장 완료!")

In [ ]:
# GGUF로 내보내기 (Ollama 호환)
model.save_pretrained_gguf(
    "adhd_coach_gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

print("GGUF 내보내기 완료!")

In [ ]:
# 파일 다운로드 (GGUF)
from google.colab import files
import os

# GGUF 파일 찾기
gguf_files = [f for f in os.listdir("adhd_coach_gguf") if f.endswith(".gguf")]
if gguf_files:
    print(f"다운로드할 파일: {gguf_files[0]}")
    files.download(f"adhd_coach_gguf/{gguf_files[0]}")
else:
    print("GGUF 파일을 찾을 수 없습니다.")

## 9. Ollama에서 사용하기

다운로드한 GGUF 파일을 로컬에서 사용하려면:

1. Ollama 설치: https://ollama.ai

2. Modelfile 생성:
```
FROM ./adhd_coach.gguf

SYSTEM """당신은 ADHD 전문 코치입니다. 피코치의 말에 공감하고,
ADHD 뇌의 특성을 이해하며, 작고 구체적인 실행 단계를 제안합니다.
완벽보다 진행을, 비판보다 격려를 우선합니다."""

PARAMETER temperature 0.7
PARAMETER num_ctx 2048
```

3. 모델 등록:
```bash
ollama create adhd-coach -f Modelfile
```

4. 사용:
```bash
ollama run adhd-coach
```